# MLP optimizado con búsqueda de hiperparámetros

Mantiene MLP puro. Cambios aplicados, en corto:

- `macro_f1` como métrica principal.
- `CrossEntropyLoss` con pesos por clase.
- `AdamW` + `weight_decay`.
- Scheduler `ReduceLROnPlateau`.
- MLP con `LayerNorm`, `GELU` y `Dropout`.
- `gradient clipping`.
- Random search reproducible.
- MLflow loguea modelo solo al final de cada trial.
- `classification_report` solo al final.


In [1]:
# Imports base
import os
import copy
import json
import random
import itertools
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt
import torchvision.utils as vutils
from torch.utils.tensorboard import SummaryWriter

from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

import mlflow
import mlflow.pytorch

/home/rami/miniconda3/envs/skin-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/rami/miniconda3/envs/skin-env/lib/python3.12/site-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.7'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
# Reproducibilidad
SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [3]:
# Paths
train_dir = "data/Split_smol/train"
val_dir = "data/Split_smol/val"

# Experimentos
mlflow.set_experiment("Clasificador_Imagenes_MLP_Optimizado")
base_log_dir = "runs/mlp_optimizado"
os.makedirs(base_log_dir, exist_ok=True)
os.makedirs("artifacts", exist_ok=True)

2026/06/20 20:14:08 INFO mlflow.tracking.fluent: Experiment with name 'Clasificador_Imagenes_MLP_Optimizado' does not exist. Creating a new experiment.


In [4]:
# Dataset con encoder compartido train/val
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None, classes=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

        if classes is None:
            self.classes = sorted([p.name for p in self.root_dir.iterdir() if p.is_dir()])
        else:
            self.classes = list(classes)

        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.samples = []
        self.labels = []

        for cls_name in self.classes:
            cls_dir = self.root_dir / cls_name
            if not cls_dir.exists():
                continue
            for path in cls_dir.rglob("*"):
                if path.suffix.lower() in self.extensions:
                    label = self.class_to_idx[cls_name]
                    self.samples.append((str(path), label))
                    self.labels.append(label)

        if len(self.samples) == 0:
            raise ValueError(f"No se encontraron imágenes en: {self.root_dir}")

        # Compatibilidad con código previo.
        self.label_encoder = SimpleNamespace(classes_=np.array(self.classes))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        image = np.array(image)

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        return image, label

In [5]:
# MLP puro: Linear + LayerNorm + GELU + Dropout
class MLPClassifier(nn.Module):
    def __init__(self, num_classes, input_size, hidden_sizes=(1024, 512), dropout=0.35):
        super().__init__()

        layers = []
        in_features = input_size

        for hidden in hidden_sizes:
            layers.append(nn.Linear(in_features, hidden))
            layers.append(nn.LayerNorm(hidden))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout))
            in_features = hidden

        layers.append(nn.Linear(in_features, num_classes))
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        return self.net(x)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [6]:
# Transforms: normalización explícita + augmentations simples
def build_transforms(input_size, hflip, vflip, rb_contrast):
    train_transform = A.Compose([
        A.Resize(input_size, input_size),
        A.HorizontalFlip(p=hflip),
        A.VerticalFlip(p=vflip),
        A.RandomBrightnessContrast(p=rb_contrast),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

    val_transform = A.Compose([
        A.Resize(input_size, input_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

    return train_transform, val_transform


def build_loaders(hparams):
    train_transform, val_transform = build_transforms(
        input_size=hparams["input_size"],
        hflip=hparams["HFlip"],
        vflip=hparams["VFlip"],
        rb_contrast=hparams["RBContrast"],
    )

    train_dataset = CustomImageDataset(hparams["train_dir"], transform=train_transform)
    val_dataset = CustomImageDataset(
        hparams["val_dir"],
        transform=val_transform,
        classes=train_dataset.classes,  # mismo encoder
    )

    assert np.array_equal(
        train_dataset.label_encoder.classes_,
        val_dataset.label_encoder.classes_,
    ), "Train y val tienen clases codificadas distinto"

    num_workers = hparams.get("num_workers", 0)
    pin_memory = torch.cuda.is_available()

    train_loader = DataLoader(
        train_dataset,
        batch_size=hparams["batch_size"],
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=hparams["batch_size"],
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    return train_dataset, val_dataset, train_loader, val_loader

In [7]:
# Evaluación: macro F1 manda
@torch.no_grad()
def evaluate(model, loader, criterion, device, writer=None, epoch=None, prefix="val"):
    model.eval()

    losses = []
    all_preds = []
    all_labels = []

    for i, (images, labels) in enumerate(loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        preds = outputs.argmax(dim=1)

        losses.append(loss.item())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        # Log visual mínimo.
        if writer is not None and epoch is not None and i == 0:
            img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
            writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)

    avg_loss = float(np.mean(losses))
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    balanced_acc = balanced_accuracy_score(all_labels, all_preds)

    if writer is not None and epoch is not None:
        writer.add_scalar(f"{prefix}/loss", avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc, epoch)
        writer.add_scalar(f"{prefix}/macro_f1", macro_f1, epoch)
        writer.add_scalar(f"{prefix}/balanced_accuracy", balanced_acc, epoch)

    return avg_loss, acc, macro_f1, balanced_acc, np.array(all_labels), np.array(all_preds)

In [8]:
# Reporte final: solo para el mejor modelo del trial
def log_final_report(y_true, y_pred, classes, trial_id):
    prefix = f"trial_{trial_id}"

    report = classification_report(
        y_true,
        y_pred,
        target_names=classes,
        zero_division=0,
    )

    report_path = f"artifacts/classification_report_{prefix}.txt"
    with open(report_path, "w") as f:
        f.write(report)
    mlflow.log_artifact(report_path)

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7, 7))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, xticks_rotation=45, values_format="d")
    ax.set_title(f"Confusion Matrix - {prefix}")
    plt.tight_layout()

    cm_path = f"artifacts/confusion_matrix_{prefix}.png"
    fig.savefig(cm_path, dpi=150)
    plt.close(fig)
    mlflow.log_artifact(cm_path)

    return report

In [9]:
# Búsqueda reproducible, sin np.random.rand() por loop
hparams_space = {
    "input_size": [64, 96],
    "batch_size": [32, 64],
    "lr": [3e-4, 1e-4, 3e-5],
    "weight_decay": [1e-5, 1e-4, 1e-3],
    "hidden_sizes": [(512, 256), (1024, 512), (1024, 512, 256)],
    "dropout": [0.2, 0.35, 0.5],
    "HFlip": [0.5],
    "VFlip": [0.0, 0.5],
    "RBContrast": [0.3, 0.5],
}

base_hparams = {
    "model": "MLPClassifier",
    "loss_fn": "CrossEntropyLoss_weighted_label_smoothing",
    "train_dir": train_dir,
    "val_dir": val_dir,
    "epochs": 200,
    "es_patience": 12,
    "label_smoothing": 0.05,
    "grad_clip_norm": 1.0,
    "scheduler_factor": 0.5,
    "scheduler_patience": 4,
    "num_workers": 0,
}


def sample_configs(space, n_trials=60, seed=42):
    keys = list(space.keys())
    values = [space[k] for k in keys]

    configs = []
    for combo in itertools.product(*values):
        hparams = copy.deepcopy(base_hparams)
        hparams.update(dict(zip(keys, combo)))
        configs.append(hparams)

    rng = random.Random(seed)
    rng.shuffle(configs)
    return configs[:min(n_trials, len(configs))]

selected_configs = sample_configs(hparams_space, n_trials=60, seed=SEED)
print("trials seleccionados:", len(selected_configs))

trials seleccionados: 60


In [10]:
# Un trial completo

def train_one_config(hparams, trial_id):
    set_seed(SEED + trial_id)

    train_dataset, val_dataset, train_loader, val_loader = build_loaders(hparams)
    classes = train_dataset.label_encoder.classes_
    num_classes = len(classes)
    input_dim = hparams["input_size"] ** 2 * 3

    model = MLPClassifier(
        num_classes=num_classes,
        input_size=input_dim,
        hidden_sizes=hparams["hidden_sizes"],
        dropout=hparams["dropout"],
    ).to(device)

    # Peso por clase para evitar colapso a clases frecuentes.
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_classes),
        y=np.array(train_dataset.labels),
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=hparams["label_smoothing"],
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=hparams["lr"],
        weight_decay=hparams["weight_decay"],
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=hparams["scheduler_factor"],
        patience=hparams["scheduler_patience"],
    )

    hparams = copy.deepcopy(hparams)
    hparams["hidden_sizes"] = str(hparams["hidden_sizes"])
    hparams["count_params"] = count_parameters(model)
    hparams["num_classes"] = num_classes
    hparams["train_samples"] = len(train_dataset)
    hparams["val_samples"] = len(val_dataset)

    run_name = f"mlp_trial_{trial_id:03d}"
    writer = SummaryWriter(log_dir=os.path.join(base_log_dir, run_name))

    best_state = None
    best_epoch = -1
    best_val_macro_f1 = -1.0
    best_metrics = {}

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(hparams)

        for epoch in tqdm(range(hparams["epochs"]), desc=run_name):
            model.train()
            train_losses = []
            train_preds = []
            train_labels = []

            for images, labels in train_loader:
                images = images.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()

                # Clipping para estabilizar MLP grande.
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=hparams["grad_clip_norm"],
                )

                optimizer.step()

                preds = outputs.argmax(dim=1)
                train_losses.append(loss.item())
                train_preds.extend(preds.detach().cpu().numpy())
                train_labels.extend(labels.detach().cpu().numpy())

            train_loss = float(np.mean(train_losses))
            train_acc = accuracy_score(train_labels, train_preds)
            train_macro_f1 = f1_score(train_labels, train_preds, average="macro", zero_division=0)

            val_loss, val_acc, val_macro_f1, val_bal_acc, y_true, y_pred = evaluate(
                model=model,
                loader=val_loader,
                criterion=criterion,
                device=device,
                writer=writer,
                epoch=epoch,
                prefix="val",
            )

            scheduler.step(val_macro_f1)
            current_lr = optimizer.param_groups[0]["lr"]

            writer.add_scalar("train/loss", train_loss, epoch)
            writer.add_scalar("train/accuracy", train_acc, epoch)
            writer.add_scalar("train/macro_f1", train_macro_f1, epoch)
            writer.add_scalar("train/lr", current_lr, epoch)

            mlflow.log_metrics({
                "train_loss": train_loss,
                "train_accuracy": train_acc,
                "train_macro_f1": train_macro_f1,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
                "val_macro_f1": val_macro_f1,
                "val_balanced_accuracy": val_bal_acc,
                "lr": current_lr,
            }, step=epoch)

            # Selección por macro F1, no por accuracy.
            if val_macro_f1 > best_val_macro_f1:
                best_val_macro_f1 = val_macro_f1
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())
                best_metrics = {
                    "best_epoch": best_epoch,
                    "best_train_loss": train_loss,
                    "best_train_accuracy": train_acc,
                    "best_train_macro_f1": train_macro_f1,
                    "best_val_loss": val_loss,
                    "best_val_accuracy": val_acc,
                    "best_val_macro_f1": val_macro_f1,
                    "best_val_balanced_accuracy": val_bal_acc,
                }

            elif epoch > best_epoch + hparams["es_patience"]:
                break

        model.load_state_dict(best_state)

        # Reporte final del mejor modelo.
        val_loss, val_acc, val_macro_f1, val_bal_acc, y_true, y_pred = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            writer=None,
            epoch=None,
            prefix="val_final",
        )

        report = log_final_report(y_true, y_pred, classes, trial_id)

        # Log del modelo una sola vez por trial.
        model_path = f"artifacts/mlp_trial_{trial_id:03d}.pth"
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": list(classes),
            "hparams": hparams,
            "metrics": best_metrics,
        }, model_path)
        mlflow.log_artifact(model_path)

        input_example = torch.zeros(
            1, 3, hparams["input_size"], hparams["input_size"]
        )

        try:
            mlflow.pytorch.log_model(
                model,
                artifact_path="pytorch_model",
                input_example=input_example,
                pip_requirements=["torch", "cloudpickle"],
            )
        except Exception as e:
            print("MLflow no pudo inferir input_example; logueo sin ejemplo:", repr(e))
            mlflow.pytorch.log_model(
                model,
                artifact_path="pytorch_model",
                pip_requirements=["torch", "cloudpickle"],
            )

        mlflow.log_metrics(best_metrics)

    writer.close()

    result = copy.deepcopy(hparams)
    result.update(best_metrics)
    result["trial_id"] = trial_id
    return result

In [11]:
# Ejecutar búsqueda
results = []

for trial_id, hparams in enumerate(selected_configs):
    print(f"\nTrial {trial_id}/{len(selected_configs)-1}: {hparams}")
    result = train_one_config(hparams, trial_id)
    results.append(result)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("best_val_macro_f1", ascending=False)
results_df.to_csv("mlp_hp_results.csv", index=False)

results_df.head(10)


Trial 0/59: {'model': 'MLPClassifier', 'loss_fn': 'CrossEntropyLoss_weighted_label_smoothing', 'train_dir': 'data/Split_smol/train', 'val_dir': 'data/Split_smol/val', 'epochs': 200, 'es_patience': 12, 'label_smoothing': 0.05, 'grad_clip_norm': 1.0, 'scheduler_factor': 0.5, 'scheduler_patience': 4, 'num_workers': 0, 'input_size': 64, 'batch_size': 64, 'lr': 0.0001, 'weight_decay': 1e-05, 'hidden_sizes': (512, 256), 'dropout': 0.5, 'HFlip': 0.5, 'VFlip': 0.5, 'RBContrast': 0.3}


mlp_trial_000:  32%|███▏      | 64/200 [02:45<05:51,  2.59s/it]


MLflow no pudo inferir input_example; logueo sin ejemplo: MlflowException("Expected one of the following types:\n- pandas.DataFrame\n- numpy.ndarray\n- dictionary of (name -> numpy.ndarray)\n- scipy.sparse.csr_matrix\n- scipy.sparse.csc_matrix\n- dict\n- list\n- scalars\n- datetime.datetime\n- pydantic model instance\nbut got '<class 'torch.Tensor'>'")


2026/06/20 20:16:55 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



Trial 1/59: {'model': 'MLPClassifier', 'loss_fn': 'CrossEntropyLoss_weighted_label_smoothing', 'train_dir': 'data/Split_smol/train', 'val_dir': 'data/Split_smol/val', 'epochs': 200, 'es_patience': 12, 'label_smoothing': 0.05, 'grad_clip_norm': 1.0, 'scheduler_factor': 0.5, 'scheduler_patience': 4, 'num_workers': 0, 'input_size': 64, 'batch_size': 32, 'lr': 0.0001, 'weight_decay': 0.001, 'hidden_sizes': (1024, 512, 256), 'dropout': 0.5, 'HFlip': 0.5, 'VFlip': 0.0, 'RBContrast': 0.3}


...

mlp_trial_058:  24%|██▎       | 47/200 [02:07<06:56,  2.72s/it]
2026/06/20 22:12:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


MLflow no pudo inferir input_example; logueo sin ejemplo: MlflowException("Expected one of the following types:\n- pandas.DataFrame\n- numpy.ndarray\n- dictionary of (name -> numpy.ndarray)\n- scipy.sparse.csr_matrix\n- scipy.sparse.csc_matrix\n- dict\n- list\n- scalars\n- datetime.datetime\n- pydantic model instance\nbut got '<class 'torch.Tensor'>'")

Trial 59/59: {'model': 'MLPClassifier', 'loss_fn': 'CrossEntropyLoss_weighted_label_smoothing', 'train_dir': 'data/Split_smol/train', 'val_dir': 'data/Split_smol/val', 'epochs': 200, 'es_patience': 12, 'label_smoothing': 0.05, 'grad_clip_norm': 1.0, 'scheduler_factor': 0.5, 'scheduler_patience': 4, 'num_workers': 0, 'input_size': 96, 'batch_size': 64, 'lr': 0.0003, 'weight_decay': 0.001, 'hidden_sizes': (1024, 512, 256), 'dropout': 0.35, 'HFlip': 0.5, 'VFlip': 0.0, 'RBContrast': 0.5}


mlp_trial_059:  20%|██        | 40/200 [01:49<07:17,  2.74s/it]
2026/06/20 22:14:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


MLflow no pudo inferir input_example; logueo sin ejemplo: MlflowException("Expected one of the following types:\n- pandas.DataFrame\n- numpy.ndarray\n- dictionary of (name -> numpy.ndarray)\n- scipy.sparse.csr_matrix\n- scipy.sparse.csc_matrix\n- dict\n- list\n- scalars\n- datetime.datetime\n- pydantic model instance\nbut got '<class 'torch.Tensor'>'")


,model,loss_fn,train_dir,val_dir,epochs,es_patience,label_smoothing,grad_clip_norm,scheduler_factor,scheduler_patience,...,val_samples,best_epoch,best_train_loss,best_train_accuracy,best_train_macro_f1,best_val_loss,best_val_accuracy,best_val_macro_f1,best_val_balanced_accuracy,trial_id
53,MLPClassifier,CrossEntropyLoss_weighted_label_smoothing,data/Split_smol/train,data/Split_smol/val,200,12,0.05,1.0,0.5,4,...,165,55,0.908414,0.736377,0.741177,1.352021,0.630303,0.640294,0.640115,53
47,MLPClassifier,CrossEntropyLoss_weighted_label_smoothing,data/Split_smol/train,data/Split_smol/val,200,12,0.05,1.0,0.5,4,...,165,42,0.871945,0.751105,0.752486,1.297515,0.618182,0.623672,0.629004,47
36,MLPClassifier,CrossEntropyLoss_weighted_label_smoothing,data/Split_smol/train,data/Split_smol/val,200,12,0.05,1.0,0.5,4,...,165,33,1.089881,0.645066,0.649699,1.303618,0.612121,0.621987,0.623449,36
13,MLPClassifier,CrossEntropyLoss_weighted_label_smoothing,data/Split_smol/train,data/Split_smol/val,200,12,0.05,1.0,0.5,4,...,165,32,1.151246,0.602356,0.606922,1.197238,0.606061,0.620411,0.620274,13
46,MLPClassifier,CrossEntropyLoss_weighted_label_smoothing,data/Split_smol/train,data/Split_smol/val,200,12,0.05,1.0,0.5,4,...,165,39,0.882010,0.740795,0.745216,1.256691,0.612121,0.619831,0.618903,46
32,MLPClassifier,CrossEntropyLoss_weighted_label_smoothing,data/Split_smol/train,data/Split_smol/val,200,12,0.05,1.0,0.5,4,...,165,16,1.004029,0.703976,0.709123,1.225785,0.612121,0.618482,0.623665,32
51,MLPClassifier,CrossEntropyLoss_weighted_label_smoothing,data/Split_smol/train,data/Split_smol/val,200,12,0.05,1.0,0.5,4,...,165,41,1.044510,0.656848,0.658726,1.339231,0.600000,0.617308,0.612554,51
8,MLPClassifier,CrossEntropyLoss_weighted_label_smoothing,data/Split_smol/train,data/Split_smol/val,200,12,0.05,1.0,0.5,4,...,165,37,1.025770,0.661267,0.664206,1.311982,0.600000,0.615819,0.610173,8
59,MLPClassifier,CrossEntropyLoss_weighted_label_smoothing,data/Split_smol/train,data/Split_smol/val,200,12,0.05,1.0,0.5,4,...,165,27,1.164515,0.577320,0.577700,1.312926,0.606061,0.614901,0.615729,59
54,MLPClassifier,CrossEntropyLoss_weighted_label_smoothing,data/Split_smol/train,data/Split_smol/val,200,12,0.05,1.0,0.5,4,...,165,44,1.145206,0.597938,0.603632,1.352715,0.606061,0.613434,0.613348,54


In [12]:
# Mejor configuración encontrada
best = results_df.iloc[0].to_dict()
print("Mejor trial:", int(best["trial_id"]))
print("Best val macro F1:", best["best_val_macro_f1"])
print("Best val accuracy:", best["best_val_accuracy"])
print("Best epoch:", int(best["best_epoch"]))

best

Mejor trial: 53
Best val macro F1: 0.6402939254508959
Best val accuracy: 0.6303030303030303
Best epoch: 55


{'model': 'MLPClassifier',
 'loss_fn': 'CrossEntropyLoss_weighted_label_smoothing',
 'train_dir': 'data/Split_smol/train',
 'val_dir': 'data/Split_smol/val',
 'epochs': 200,
 'es_patience': 12,
 'label_smoothing': 0.05,
 'grad_clip_norm': 1.0,
 'scheduler_factor': 0.5,
 'scheduler_patience': 4,
 'num_workers': 0,
 'input_size': 64,
 'batch_size': 64,
 'lr': 0.0001,
 'weight_decay': 0.001,
 'hidden_sizes': '(1024, 512)',
 'dropout': 0.35,
 'HFlip': 0.5,
 'VFlip': 0.5,
 'RBContrast': 0.3,
 'count_params': 13116425,
 'num_classes': 9,
 'train_samples': 679,
 'val_samples': 165,
 'best_epoch': 55,
 'best_train_loss': 0.9084136702797629,
 'best_train_accuracy': 0.7363770250368189,
 'best_train_macro_f1': 0.7411772087420659,
 'best_val_loss': 1.352020502090454,
 'best_val_accuracy': 0.6303030303030303,
 'best_val_macro_f1': 0.6402939254508959,
 'best_val_balanced_accuracy': 0.6401154401154401,
 'trial_id': 53}

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/mlp_optimizado --port 6006